In [1]:
# Cell 1 — pull and merge Akola Jassid + Aphid, 2005-2009
import pandas as pd, requests, time
from io import StringIO

def pull(pest):
    url = "http://icar-crida.res.in:8080/naip/Servlet1.jsp"
    params = {"combo0": "Cotton", "combo1": "Akola", "combo2": pest,
              "combo3": "2005", "combo4": "2009"}
    resp = requests.get(url, params=params, timeout=15)
    return pd.read_html(StringIO(resp.text))[2]

jassid = pull("Jassid").rename(columns={"Pest Value": "Jassid_Value"})
time.sleep(1)
aphid = pull("Aphid").rename(columns={"Pest Value": "Aphid_Value"})

df = jassid.merge(
    aphid[["Observation Year", "Standard Week", "Aphid_Value"]],
    on=["Observation Year", "Standard Week"], how="outer"
).sort_values(["Observation Year", "Standard Week"]).reset_index(drop=True)
print(df.shape, df.columns.tolist())

(260, 12) ['Observation Year', 'Standard Week', 'Jassid_Value', 'MaxT(°C)', 'MinT(°C)', 'RH1(%)', 'RH2(%)', 'RF(mm)', 'WS(kmph)', 'SSH(hrs)', 'EVP(mm)', 'Aphid_Value']


In [2]:
# Cell 2 — impute within-season, normalize, combine into one pest_pressure target
for col in ["Jassid_Value", "Aphid_Value"]:
    df[col] = df.groupby("Observation Year")[col].transform(lambda x: x.interpolate())
df = df.dropna(subset=["Jassid_Value", "Aphid_Value"]).reset_index(drop=True)

# min-max scale each pest column using ONLY train years (2005-2008) to avoid leakage
train_mask = df["Observation Year"] <= 2008
for col in ["Jassid_Value", "Aphid_Value"]:
    lo, hi = df.loc[train_mask, col].min(), df.loc[train_mask, col].max()
    df[col + "_norm"] = ((df[col] - lo) / (hi - lo)).clip(0, 1)

df["pest_pressure"] = df[["Jassid_Value_norm", "Aphid_Value_norm"]].max(axis=1)
print(df[["Observation Year", "Standard Week", "pest_pressure"]].head())

   Observation Year  Standard Week  pest_pressure
0              2005             31       0.019048
1              2005             32       0.169082
2              2005             33       0.785507
3              2005             34       0.102899
4              2005             35       0.247619


In [3]:
# Cell 3 — build sliding windows (4-week lookback) per season, split by year
import numpy as np

weather_cols = ["MaxT(°C)", "MinT(°C)", "RH1(%)", "RH2(%)", "RF(mm)", "WS(kmph)", "SSH(hrs)", "EVP(mm)"]
feature_cols = weather_cols + ["pest_pressure"]
WINDOW = 4

def make_sequences(sub_df):
    X, y = [], []
    vals = sub_df[feature_cols].values
    target = sub_df["pest_pressure"].values
    for i in range(WINDOW, len(vals)):
        X.append(vals[i-WINDOW:i])
        y.append(target[i])
    return np.array(X), np.array(y)

train_df = df[df["Observation Year"] <= 2008]
test_df  = df[df["Observation Year"] == 2009]

X_train, y_train = make_sequences(train_df)
X_test,  y_test  = make_sequences(test_df)
print(X_train.shape, X_test.shape)

(117, 4, 9) (19, 4, 9)


In [4]:
# Cell 4 — small LSTM, regression head
from tensorflow.keras import layers, models, callbacks

model = models.Sequential([
    layers.Input(shape=(WINDOW, len(feature_cols))),
    layers.LSTM(16, dropout=0.3),
    layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                     epochs=50, batch_size=8, callbacks=[es], verbose=1)

Epoch 1/50


2026-09-19 08:20:42.004658: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.0657 - mae: 0.2065 - val_loss: 0.0879 - val_mae: 0.2022
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0655 - mae: 0.2103 - val_loss: 0.0817 - val_mae: 0.1972
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0502 - mae: 0.1793 - val_loss: 0.0830 - val_mae: 0.1952
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0462 - mae: 0.1711 - val_loss: 0.0894 - val_mae: 0.1987
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0478 - mae: 0.1644 - val_loss: 0.0932 - val_mae: 0.2009
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0547 - mae: 0.1815 - val_loss: 0.0935 - val_mae: 0.2007
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0451 - mae: 0.1628 - val_loss: 0.0891 - val_mae: 0.1961


In [5]:
# Cell 5 — evaluate
preds = model.predict(X_test).flatten()
mae = np.mean(np.abs(preds - y_test))
print(f"Test MAE: {mae:.4f}")
print("Sample predictions vs actual:", list(zip(preds[:10].round(3), y_test[:10].round(3))))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step
Test MAE: 0.1972
Sample predictions vs actual: [(np.float32(0.354), np.float64(1.0)), (np.float32(0.351), np.float64(0.424)), (np.float32(0.402), np.float64(1.0)), (np.float32(0.384), np.float64(1.0)), (np.float32(0.225), np.float64(0.371)), (np.float32(0.19), np.float64(0.157)), (np.float32(0.309), np.float64(0.138)), (np.float32(0.375), np.float64(0.167)), (np.float32(0.198), np.float64(0.281)), (np.float32(0.198), np.float64(0.2))]


In [6]:
# Cell 6 — does the LSTM beat trivial guessing?
mean_baseline = np.full_like(y_test, y_train.mean())
persistence_baseline = X_test[:, -1, -1]  # last known pest_pressure value per window

print("LSTM MAE:         ", mae)
print("Mean-baseline MAE: ", np.mean(np.abs(mean_baseline - y_test)))
print("Persistence MAE:   ", np.mean(np.abs(persistence_baseline - y_test)))

LSTM MAE:          0.19721378498806388
Mean-baseline MAE:  0.20557848001045756
Persistence MAE:    0.15388471177944862
